# BUSINFO 701 – Business Analytics Tools

### Individual Assignment 2 (Task 2)
### Student ID: 
## Bank Loan Data Analysis Using Python (Pandas and XML Integration)



The dataset contains information about bank loan applications, including customer details such as account balances, loan purpose, employment duration, education level, and credit risk. The data was imported from a text file (bank.txt) into a pandas DataFrame for analysis.

The task involves several key steps:

Data Loading and Cleaning – reading the dataset into Python, removing metadata, and converting relevant columns (like checking and savings balances) to numeric types.

Feature Creation – adding a new column, TotalBalance, by summing the checking and savings account balances without altering existing data types.

Sorting and Filtering – identifying the top 20 customers with the lowest total balances, with ties resolved using customer numbers in ascending order.

Data Analysis – answering questions such as identifying the customer number for specific observations and calculating the difference between selected total balances.


### Task 2 Question 1 - Data Loading and Inspection

In [47]:
# Load the CSV file's data into a DataFrame named bank_loans

import pandas as pd

# Load CSV/DSV into a DataFrame 
bank_loans = pd.read_csv("bank.txt", sep=",", skiprows=23)



The dataset bank.txt was loaded into a pandas DataFrame named bank_loans using the read_csv() function.
The parameter sep="," specifies that the file is comma-separated, and skiprows=23 skips metadata lines to start reading from the actual header row.

#### Inspecting the dataset

In [80]:
# Use appropriate Python command(s) to inspect bank_loans 
# Using head to preview first few rows and to validate skip rows value 
bank_loans.head()          


,CustomerNo,ApplicationDate,LoanPurpose,Checking,Savings,MonthsCustomer,MonthsEmployed,EducationLevel,Gender,MaritalStatus,...,Housing,Years,Job,CreditRisk,Checking_num,Savings_num,TotalBalance,AppDate,ApplicationMonth,requires_censored
0,1,22/10/23,Repairs,$207,$0,28,116,Doctorate,M,Single,...,Own,4,Skilled,Low,207.0,0.0,207.0,2023-10-22,Oct,False
1,2,27/1/23,New Car,$193,"$2,684",13,5,Master's Degree,F,Divorced,...,Own,2,Unskilled,High,193.0,2684.0,2877.0,2023-01-27,Jan,False
2,3,21/8/23,Small Appliance,$0,$739,13,12,Associate Degree,M,Single,...,Own,3,Unskilled,Low,0.0,739.0,739.0,2023-08-21,Aug,True
3,4,10/1/23,Furniture,$0,"$1,230",25,0,Associate Degree,M,Divorced,...,Own,1,Skilled,High,0.0,1230.0,1230.0,2023-01-10,Jan,True
4,5,11/4/23,New Car,$0,$389,19,119,High School,M,Single,...,Own,4,Management,High,0.0,389.0,389.0,2023-04-11,Apr,False


In [81]:
# Using info() to view structure & dtypes
bank_loans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 21 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   CustomerNo         418 non-null    int64         
 1   ApplicationDate    418 non-null    object        
 2   LoanPurpose        418 non-null    object        
 3   Checking           418 non-null    object        
 4   Savings            418 non-null    object        
 5   MonthsCustomer     418 non-null    int64         
 6   MonthsEmployed     418 non-null    int64         
 7   EducationLevel     418 non-null    object        
 8   Gender             418 non-null    object        
 9   MaritalStatus      418 non-null    object        
 10  Age                418 non-null    int64         
 11  Housing            418 non-null    object        
 12  Years              418 non-null    int64         
 13  Job                418 non-null    object        
 14  CreditRisk

The dataset is structured and readable after skipping metadata rows. Column types are consistent with variable meaning (balances as text to be cleaned, dates as strings). No missing values were observed in the key columns from .info(). The file is therefore suitable for cleaning and analysis using the lecture methods.

In [82]:
# Using describe() to view statistics
bank_loans.describe()

,CustomerNo,MonthsCustomer,MonthsEmployed,Age,Years,Checking_num,Savings_num,TotalBalance,AppDate
count,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418.000000,418
mean,210.193780,22.894737,31.559809,34.342105,2.832536,1062.466507,1836.717703,2899.184211,2023-07-02 03:54:15.502392320
min,1.000000,5.000000,0.000000,18.000000,1.000000,0.000000,0.000000,0.000000,2023-01-01 00:00:00
25%,105.250000,13.000000,6.000000,26.000000,2.000000,0.000000,230.750000,490.750000,2023-03-28 18:00:00
50%,209.500000,19.000000,20.000000,32.000000,3.000000,0.000000,600.500000,836.000000,2023-07-03 00:00:00
75%,313.750000,28.000000,46.000000,41.000000,4.000000,563.750000,921.750000,2676.250000,2023-10-07 12:00:00
max,425.000000,73.000000,119.000000,73.000000,4.000000,19812.000000,19811.000000,32542.000000,2023-12-30 00:00:00
std,121.875623,12.196765,32.254496,11.089788,1.084309,3171.241711,3622.181816,4857.832421,NaN


In [83]:
# Inspecting total rows and columns
bank_loans.shape

(418, 21)

The dataset contains 418 applications and 15 variables. This confirms the cleaning step correctly skipped the 23 metadata rows.

**How many loan applications are there in total?**
There are 418 loan applications in total.

This was determined using the .shape function, which returns the number of rows and columns in the dataset, each row representing one loan application. Pandas automatically excludes the header row when importing data, treating the first line (CustomerNo, ApplicationDate, ...) as column names rather than a data record.

**How many columns are there in total? Which column(s) contain integer values? Which column(s) contain string/text values?**
There are 15 columns in total, consisting of 5 integer columns and 10 string (object) columns.
This information was obtained using the .info() function, which displays each column’s data type and non-null count.

Integer columns (int64): CustomerNo, MonthsCustomer, MonthsEmployed, Age, Years

String/Text columns (object): ApplicationDate, LoanPurpose, Checking, Savings, EducationLevel, Gender, MaritalStatus, Housing, Job, CreditRisk

### Task 2 Question 2 - Column creation, String cleaning, and Arithmetic operations

In [84]:
# Convert string columns and add into numeric columns for arithmetic operation
bank_loans["Checking_num"] = bank_loans["Checking"].str.replace("$","").str.replace(",","").astype(float)
bank_loans["Savings_num"]  = bank_loans["Savings"].str.replace("$","").str.replace(",","").astype(float)



In [85]:
# Add a new column called TotalBalance, which sums up the checking and savings account balance
bank_loans["TotalBalance"] = bank_loans["Checking_num"] + bank_loans["Savings_num"]


In [86]:
# Sort TotalBalance by ascending order, then custom sort by CustomerNo; Use head () function to display 20 records with lowest value
lowest_20 = bank_loans.sort_values(["TotalBalance", "CustomerNo"], ascending=[True, True])

# List the customer information of the top 20 customers who have the lowest total balances
lowest_20.head(20)

,CustomerNo,ApplicationDate,LoanPurpose,Checking,Savings,MonthsCustomer,MonthsEmployed,EducationLevel,Gender,MaritalStatus,...,Housing,Years,Job,CreditRisk,Checking_num,Savings_num,TotalBalance,AppDate,ApplicationMonth,requires_censored
5,6,12/3/23,New Car,$0,$0,25,103,Associate Degree,F,Divorced,...,Own,2,Skilled,High,0.0,0.0,0.0,2023-03-12,Mar,False
25,26,4/5/23,Furniture,$0,$0,25,23,Doctorate,M,Married,...,Own,4,Skilled,High,0.0,0.0,0.0,2023-05-04,May,True
29,30,3/2/23,Used Car,$0,$0,19,58,Associate Degree,M,Single,...,Other,4,Skilled,High,0.0,0.0,0.0,2023-02-03,Feb,False
58,59,10/8/23,Education,$0,$0,37,114,Doctorate,M,Single,...,Own,4,Management,High,0.0,0.0,0.0,2023-08-10,Aug,False
73,74,18/3/23,New Car,$0,$0,22,9,Master's Degree,M,Single,...,Own,2,Unskilled,High,0.0,0.0,0.0,2023-03-18,Mar,False
133,134,22/4/23,Furniture,$0,$0,13,94,High School,M,Single,...,Rent,4,Skilled,Low,0.0,0.0,0.0,2023-04-22,Apr,True
137,138,1/2/23,Other,$0,$0,25,54,Bachelor's Degree,M,Single,...,Own,3,Management,High,0.0,0.0,0.0,2023-02-01,Feb,False
194,195,12/4/23,New Car,$0,$0,25,19,Master's Degree,F,Divorced,...,Rent,4,Skilled,High,0.0,0.0,0.0,2023-04-12,Apr,False
285,286,19/8/23,Used Car,$0,$0,37,49,Associate Degree,M,Single,...,Other,4,Skilled,High,0.0,0.0,0.0,2023-08-19,Aug,False
290,291,3/2/23,Small Appliance,$0,$0,43,28,High School,F,Divorced,...,Own,3,Management,High,0.0,0.0,0.0,2023-02-03,Feb,True


In [87]:
# Sixth data record customer number using iloc:

sixth_customer_no = lowest_20.iloc[5]["CustomerNo"]
print("The sixth customer number is:", int(sixth_customer_no))


The sixth customer number is: 134


In [102]:
# Difference between the lowest and sixteenth-lowest total balances:
diff_in_Totalbalance = lowest_20.iloc[15]["TotalBalance"] - lowest_20.iloc[0]["TotalBalance"]
Totbal_16thlowest =  lowest_20.iloc[15]["TotalBalance"]
Totbal_lowest = lowest_20.iloc[0]["TotalBalance"]

print("The total balance for 16th lowest customer is:", int(Totbal_16thlowest))
print("The lowest total balance value is:", int(Totbal_lowest))

print("The total balance difference for 16th lowest customer & lowest customer is:", int(diff_in_Totalbalance))


The total balance for 16th lowest customer is: 108
The lowest total balance value is: 0
The total balance difference for 16th lowest customer & lowest customer is: 108


Column creation from string-cleaned fields and arithmetic, followed by multi-key sorting and positional selection

**Which is the customer number for the sixth observation? Identify the customer number?**
The sixth customer number is 134

The .iloc[] function was used to locate the sixth row (index position 5) in the sorted dataset.
After sorting by total balance and customer number using .sort_values(), .iloc[5]["CustomerNo"] retrieved the customer number at that position.

**What is the difference in value between the lowest and sixteenth-lowest total balances?**
The difference between the lowest and sixteenth-lowest total balances is 108.

The .iloc[] function was again used to access specific rows (index 0 and 15) representing the lowest and sixteenth-lowest total balances.
Their TotalBalance values were subtracted to find the difference.

### Task 2 Question 3 - Date formatting, Grouping, and Aggregation to extract month-level insights

The below code converts the string-based ApplicationDate values into proper datetime objects using the pd.to_datetime() function, with the format specified as day/month/year (%d/%m/%y).
This allows uniform date representation and enables further analysis, such as extracting the application month or year.

In [89]:
# Create a new column called ApplicationMonth
# Dates transformed uniformly in a specified format (dd/mm/yy)
bank_loans["AppDate"] = pd.to_datetime(bank_loans["ApplicationDate"], format="%d/%m/%y")
# Month label to be added like "Jan", "Feb"
bank_loans["ApplicationMonth"] = bank_loans["AppDate"].dt.strftime("%b")

bank_loans




,CustomerNo,ApplicationDate,LoanPurpose,Checking,Savings,MonthsCustomer,MonthsEmployed,EducationLevel,Gender,MaritalStatus,...,Housing,Years,Job,CreditRisk,Checking_num,Savings_num,TotalBalance,AppDate,ApplicationMonth,requires_censored
0,1,22/10/23,Repairs,$207,$0,28,116,Doctorate,M,Single,...,Own,4,Skilled,Low,207.0,0.0,207.0,2023-10-22,Oct,False
1,2,27/1/23,New Car,$193,"$2,684",13,5,Master's Degree,F,Divorced,...,Own,2,Unskilled,High,193.0,2684.0,2877.0,2023-01-27,Jan,False
2,3,21/8/23,Small Appliance,$0,$739,13,12,Associate Degree,M,Single,...,Own,3,Unskilled,Low,0.0,739.0,739.0,2023-08-21,Aug,True
3,4,10/1/23,Furniture,$0,"$1,230",25,0,Associate Degree,M,Divorced,...,Own,1,Skilled,High,0.0,1230.0,1230.0,2023-01-10,Jan,True
4,5,11/4/23,New Car,$0,$389,19,119,High School,M,Single,...,Own,4,Management,High,0.0,389.0,389.0,2023-04-11,Apr,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,421,2/9/23,Furniture,$0,$957,19,11,Bachelor's Degree,F,Divorced,...,Rent,4,Skilled,High,0.0,957.0,957.0,2023-09-02,Sep,True
414,422,23/3/23,New Car,$0,$770,37,3,Bachelor's Degree,F,Divorced,...,Own,4,Skilled,High,0.0,770.0,770.0,2023-03-23,Mar,False
415,423,14/3/23,Furniture,$983,$950,13,5,High School,F,Divorced,...,Rent,3,Skilled,High,983.0,950.0,1933.0,2023-03-14,Mar,True
416,424,9/1/23,Used Car,$0,$160,13,7,Master's Degree,M,Married,...,Rent,4,Skilled,Low,0.0,160.0,160.0,2023-01-09,Jan,False


The code also extracts the month from the AppDate column using the .dt accessor, which allows access to datetime properties in pandas.

The .strftime("%b") function then formats each date to display the three-letter month abbreviation (e.g., Jan, Feb, Mar).

This new column, ApplicationMonth, helps in analyzing the number of loan applications received each month.

#### The number of applications in each month.

In [90]:
# Determine the number of applications in each month
month_order = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4,
    "May": 5, "Jun": 6, "Jul": 7, "Aug": 8,
    "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}
appl_monthwise = bank_loans["ApplicationMonth"].value_counts().reset_index()
appl_monthwise.columns = ["ApplicationMonth", "Count"]
appl_monthwise["MonthNum"] = appl_monthwise["ApplicationMonth"].map(month_order)
appl_monthwise = appl_monthwise.sort_values("MonthNum")
appl_monthwise[["ApplicationMonth", "Count"]]

,ApplicationMonth,Count
5,Jan,34
8,Feb,31
1,Mar,43
0,Apr,43
11,May,24
7,Jun,32
6,Jul,33
4,Aug,37
9,Sep,31
10,Oct,30


The code counts how many loan applications were submitted in each month using the value_counts() function.

The .map() function assigns a numeric order (Jan–Dec) so the results can be sorted chronologically with .sort_values().

This helps identify seasonal trends in application volume throughout the year. March and April recorded 43 applications, the highest among all other months.

#### The number of applications for each marital status in each month.

In [91]:
# Determine the number of applications for each marital status in each month
applmonthwise_by_marital = (
    bank_loans.groupby(["ApplicationMonth", "MaritalStatus"])
    .size()
    .reset_index(name="Count")
)
applmonthwise_by_marital["MonthNum"] = applmonthwise_by_marital["ApplicationMonth"].map(month_order)
appl_by_maritalstatus = applmonthwise_by_marital.sort_values("MonthNum")
display(appl_by_maritalstatus[["ApplicationMonth", "MaritalStatus", "Count"]])


,ApplicationMonth,MaritalStatus,Count
12,Jan,Married,2
13,Jan,Single,13
11,Jan,Divorced,19
8,Feb,Divorced,14
9,Feb,Married,3
10,Feb,Single,14
22,Mar,Single,23
21,Mar,Married,2
20,Mar,Divorced,18
2,Apr,Single,31


Here, the data is grouped by both ApplicationMonth and MaritalStatus using the groupby() and .size() functions.

The .reset_index() function converts the grouped data into a readable table, while .map() and .sort_values() ensure the months appear in calendar order.

This summary shows how loan applications vary across different marital statuses each month.

#### The number of different loan purposes in each month.

In [92]:
# Determine the number of different loan purposes in each month
Loan_purpose_per_month = (
    bank_loans.groupby("ApplicationMonth")["LoanPurpose"]
    .nunique()
    .reset_index(name="LoanPurpose_MonthlyCount")
)
Loan_purpose_per_month["MonthNum"] = Loan_purpose_per_month["ApplicationMonth"].map(month_order)
Loan_purpose_monthwise = Loan_purpose_per_month.sort_values("MonthNum")
display(Loan_purpose_monthwise[["ApplicationMonth", "LoanPurpose_MonthlyCount"]])

,ApplicationMonth,LoanPurpose_MonthlyCount
4,Jan,8
3,Feb,7
7,Mar,7
0,Apr,7
8,May,5
6,Jun,7
5,Jul,6
1,Aug,8
11,Sep,8
10,Oct,8


Date parsing with a specified format (%d/%m/%y), month derivation via .dt/strftime, and aggregations using value_counts, groupby().size(), and nunique() are the techniques used.

The .map() and .sort_values() steps arrange the months from January to December.

This analysis highlights how diverse the loan purposes were across different months.

### Task 2 Question 4 - Iterative and Conditional control flows with function application to create new boolean variables

Iterative and conditional control flows in a user-defined function, applied with .apply() to create a boolean column, then filtered via a logical OR condition

In [93]:
# Create a function called check_cat_censored
CENSORED = ["furniture", "appliance"]

In [94]:
# Create a new column called requires_censored
def check_cat_censored(loan_purpose):
    text = str(loan_purpose).lower()
    # Iteration with condition to check
    for kw in CENSORED:
        if kw in text:
            return True
    return False

bank_loans["requires_censored"] = bank_loans["LoanPurpose"].apply(check_cat_censored)
bank_loans


,CustomerNo,ApplicationDate,LoanPurpose,Checking,Savings,MonthsCustomer,MonthsEmployed,EducationLevel,Gender,MaritalStatus,...,Housing,Years,Job,CreditRisk,Checking_num,Savings_num,TotalBalance,AppDate,ApplicationMonth,requires_censored
0,1,22/10/23,Repairs,$207,$0,28,116,Doctorate,M,Single,...,Own,4,Skilled,Low,207.0,0.0,207.0,2023-10-22,Oct,False
1,2,27/1/23,New Car,$193,"$2,684",13,5,Master's Degree,F,Divorced,...,Own,2,Unskilled,High,193.0,2684.0,2877.0,2023-01-27,Jan,False
2,3,21/8/23,Small Appliance,$0,$739,13,12,Associate Degree,M,Single,...,Own,3,Unskilled,Low,0.0,739.0,739.0,2023-08-21,Aug,True
3,4,10/1/23,Furniture,$0,"$1,230",25,0,Associate Degree,M,Divorced,...,Own,1,Skilled,High,0.0,1230.0,1230.0,2023-01-10,Jan,True
4,5,11/4/23,New Car,$0,$389,19,119,High School,M,Single,...,Own,4,Management,High,0.0,389.0,389.0,2023-04-11,Apr,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,421,2/9/23,Furniture,$0,$957,19,11,Bachelor's Degree,F,Divorced,...,Rent,4,Skilled,High,0.0,957.0,957.0,2023-09-02,Sep,True
414,422,23/3/23,New Car,$0,$770,37,3,Bachelor's Degree,F,Divorced,...,Own,4,Skilled,High,0.0,770.0,770.0,2023-03-23,Mar,False
415,423,14/3/23,Furniture,$983,$950,13,5,High School,F,Divorced,...,Rent,3,Skilled,High,983.0,950.0,1933.0,2023-03-14,Mar,True
416,424,9/1/23,Used Car,$0,$160,13,7,Master's Degree,M,Married,...,Rent,4,Skilled,Low,0.0,160.0,160.0,2023-01-09,Jan,False


In [95]:
# Output the number of customers who satisfy either of the following two conditions: their loan needs to be censored, or they have been with the bank for at least two years. 
count_censored_or_2yrscust = ((bank_loans["requires_censored"]) | (bank_loans["MonthsCustomer"] >= 24)).sum()
count_censored_or_2yrscust

np.int64(302)

The code first converts each loan purpose to lowercase and checks whether it contains the keywords “furniture” or “appliance.”

It then creates a new boolean column, requires_censored, which is marked True for these cases.

Finally, it counts all customers who either require censorship or have been with the bank for 24 months or longer.

**The total number of customers loan needs to be censored, or they have been with the bank for at least two years is 302.**

### Task 2 Question 5 - Pivot tables to Summarise mean values by Categorical groups

Pivot_table to compute mean values by categorical groups, rounded for presentation. 

These summaries help compare customer stability (Years) and maturity (Age) across Gender × Job, for product marketing & targeting.

In [96]:
# Generate a Pivot table to summarise the average age and the average number of years at current residence for different genders and job types
Pivot_demography = bank_loans.pivot_table(
    index="Gender", 
    columns="Job",
    values=["Age", "Years"],
    aggfunc="mean"
).round(2)

Pivot_demography

Age                                   Years                     \
Job    Management Skilled Unemployed Unskilled Management Skilled Unemployed   
Gender                                                                         
F           38.55   29.93      34.86     34.67       2.82    2.82       2.57   
M           38.56   34.35      38.25     36.60       2.91    2.80       2.75   

                  
Job    Unskilled  
Gender            
F           3.11  
M           2.82

The code groups the dataset by Gender and Job using the groupby() function, which allows summarising data by categories.

It then calculates the average Age and average Years at current residence for each combination of gender and job using the .mean() function.
The .round(2) method is applied to display the results rounded to two decimal places for clarity.

This summary helps compare customer demographics (age and years at residence) across different gender and job groups.

### Task 2 Question 6 - Dataset merging, Sorting, and Index resetting using pandas merge and sort_values

This section loads and merges the bank customer dataset (bank_loans) with the XML monthly quota dataset (mon_info). It follows the merge, sort, and reset-index operations.

The below code reads the XML file into a pandas DataFrame using read_xml().

In [97]:
# Load the XML file's data into a DataFrame named mon_info
mon_info = pd.read_xml("bank_extra.xml")
# Inspect the loaded xml file
mon_info

,Mon,MaxAppNum
0,Jan,10
1,Feb,12
2,Mar,12
3,Apr,12
4,May,15
5,Jun,15
6,Jul,15
7,Aug,15
8,Sep,12
9,Oct,12


The column name Mon is renamed to ApplicationMonth so that both DataFrames have a common key for merging.
The .str.strip() function removes unwanted spaces to avoid mismatches during the merge.

In [98]:
# Inspect Columns in uploaded xml file
mon_info.columns

# Align the join key, convert Mon -> ApplicationMonth & trim spaces
mon_info = mon_info.rename(columns={"Mon": "ApplicationMonth"})
mon_info["ApplicationMonth"] = mon_info["ApplicationMonth"].astype(str).str.strip()

mon_info


,ApplicationMonth,MaxAppNum
0,Jan,10
1,Feb,12
2,Mar,12
3,Apr,12
4,May,15
5,Jun,15
6,Jul,15
7,Aug,15
8,Sep,12
9,Oct,12


Both datasets are structured and complete, with no missing values detected in .info(). Column names follow consistent naming conventions and include numeric and text variables suitable for analysis. 

The XML file adds month information, which is reliable for aggregation since its entries match standard calendar months.


Both DataFrames share the common attribute ApplicationMonth, which provides a consistent key for merging. The text month format (Jan–Dec) aligns across datasets, ensuring compatibility in granularity and scope. pd.merge() combines both datasets based on the ApplicationMonth column. 

A left join ensures all customer applications from bank_loans are retained, even if a month in mon_info has missing records.

In [99]:
# Combine bank_loans and mon_info into one single DataFrame loans
loans = pd.merge(bank_loans, mon_info, on="ApplicationMonth", how="left")
loans

,CustomerNo,ApplicationDate,LoanPurpose,Checking,Savings,MonthsCustomer,MonthsEmployed,EducationLevel,Gender,MaritalStatus,...,Years,Job,CreditRisk,Checking_num,Savings_num,TotalBalance,AppDate,ApplicationMonth,requires_censored,MaxAppNum
0,1,22/10/23,Repairs,$207,$0,28,116,Doctorate,M,Single,...,4,Skilled,Low,207.0,0.0,207.0,2023-10-22,Oct,False,12
1,2,27/1/23,New Car,$193,"$2,684",13,5,Master's Degree,F,Divorced,...,2,Unskilled,High,193.0,2684.0,2877.0,2023-01-27,Jan,False,10
2,3,21/8/23,Small Appliance,$0,$739,13,12,Associate Degree,M,Single,...,3,Unskilled,Low,0.0,739.0,739.0,2023-08-21,Aug,True,15
3,4,10/1/23,Furniture,$0,"$1,230",25,0,Associate Degree,M,Divorced,...,1,Skilled,High,0.0,1230.0,1230.0,2023-01-10,Jan,True,10
4,5,11/4/23,New Car,$0,$389,19,119,High School,M,Single,...,4,Management,High,0.0,389.0,389.0,2023-04-11,Apr,False,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,421,2/9/23,Furniture,$0,$957,19,11,Bachelor's Degree,F,Divorced,...,4,Skilled,High,0.0,957.0,957.0,2023-09-02,Sep,True,12
414,422,23/3/23,New Car,$0,$770,37,3,Bachelor's Degree,F,Divorced,...,4,Skilled,High,0.0,770.0,770.0,2023-03-23,Mar,False,12
415,423,14/3/23,Furniture,$983,$950,13,5,High School,F,Divorced,...,3,Skilled,High,983.0,950.0,1933.0,2023-03-14,Mar,True,12
416,424,9/1/23,Used Car,$0,$160,13,7,Master's Degree,M,Married,...,4,Skilled,Low,0.0,160.0,160.0,2023-01-09,Jan,False,10


Months are text labels, a dictionary is used to assign numeric order values. This enables sorting the months chronologically instead of alphabetically.

sort_values() orders the data first by month, then by customer number.

reset_index(drop=True) reassigns new index values starting from 0, as required in the task.

Finally, the temporary helper column MonthOrder is removed to keep the DataFrame clean

In [100]:
# Sort loans by the column ApplicationMonth and then by the customer number in asc order. Then, save the sorted results in a new DataFrame sorted_loans.

month_order = {
    "Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
    "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12
}
loans["MonthOrder"] = loans["ApplicationMonth"].map(month_order)

sorted_loans = (
    loans.sort_values(["MonthOrder", "CustomerNo"])
         .reset_index(drop=True)
    .drop(columns=["MonthOrder"])
)

sorted_loans

,CustomerNo,ApplicationDate,LoanPurpose,Checking,Savings,MonthsCustomer,MonthsEmployed,EducationLevel,Gender,MaritalStatus,...,Years,Job,CreditRisk,Checking_num,Savings_num,TotalBalance,AppDate,ApplicationMonth,requires_censored,MaxAppNum
0,2,27/1/23,New Car,$193,"$2,684",13,5,Master's Degree,F,Divorced,...,2,Unskilled,High,193.0,2684.0,2877.0,2023-01-27,Jan,False,10
1,4,10/1/23,Furniture,$0,"$1,230",25,0,Associate Degree,M,Divorced,...,1,Skilled,High,0.0,1230.0,1230.0,2023-01-10,Jan,True,10
2,13,11/1/23,Small Appliance,$966,$0,25,4,High School,F,Divorced,...,1,Skilled,High,966.0,0.0,966.0,2023-01-11,Jan,True,10
3,16,10/1/23,Business,$322,$578,10,14,High School,M,Married,...,1,Skilled,Low,322.0,578.0,900.0,2023-01-10,Jan,False,10
4,23,3/1/23,Education,$287,"$12,348",7,2,Bachelor's Degree,F,Divorced,...,2,Skilled,High,287.0,12348.0,12635.0,2023-01-03,Jan,False,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,354,24/12/23,Furniture,$0,"$12,721",37,31,Bachelor's Degree,F,Divorced,...,4,Skilled,Low,0.0,12721.0,12721.0,2023-12-24,Dec,True,10
414,356,8/12/23,Repairs,"$3,111",$0,13,27,High School,F,Divorced,...,4,Skilled,Low,3111.0,0.0,3111.0,2023-12-08,Dec,False,10
415,358,20/12/23,Furniture,$0,$538,25,59,Associate Degree,M,Single,...,2,Management,High,0.0,538.0,538.0,2023-12-20,Dec,True,10
416,360,17/12/23,Small Appliance,$231,$702,10,99,Doctorate,M,Single,...,4,Unskilled,Low,231.0,702.0,933.0,2023-12-17,Dec,True,10


In [101]:
# Validating if index starts at 0

sorted_loans.head()

,CustomerNo,ApplicationDate,LoanPurpose,Checking,Savings,MonthsCustomer,MonthsEmployed,EducationLevel,Gender,MaritalStatus,...,Years,Job,CreditRisk,Checking_num,Savings_num,TotalBalance,AppDate,ApplicationMonth,requires_censored,MaxAppNum
0,2,27/1/23,New Car,$193,"$2,684",13,5,Master's Degree,F,Divorced,...,2,Unskilled,High,193.0,2684.0,2877.0,2023-01-27,Jan,False,10
1,4,10/1/23,Furniture,$0,"$1,230",25,0,Associate Degree,M,Divorced,...,1,Skilled,High,0.0,1230.0,1230.0,2023-01-10,Jan,True,10
2,13,11/1/23,Small Appliance,$966,$0,25,4,High School,F,Divorced,...,1,Skilled,High,966.0,0.0,966.0,2023-01-11,Jan,True,10
3,16,10/1/23,Business,$322,$578,10,14,High School,M,Married,...,1,Skilled,Low,322.0,578.0,900.0,2023-01-10,Jan,False,10
4,23,3/1/23,Education,$287,"$12,348",7,2,Bachelor's Degree,F,Divorced,...,2,Skilled,High,287.0,12348.0,12635.0,2023-01-03,Jan,False,10


Displays the first few rows to confirm that the merge and sorting operations were successful and that the index starts at 0.

The analysis is feasible because the CSV and XML files are structured and share a compatible merge key at the month level. If mismatches occur, backup steps are: 

(1) Standardise month labels with rename/strip/mapping

(2) For missing quotas, retain all applications via left join and report blanks as “no quota provided” 

(3) Ensure date parsing uses the explicit %d/%m/%y format to avoid ambiguity. Performed various techniques (loading, inspection, string cleaning, date parsing, aggregation, pivoting, merging, sorting, index reset)